In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [ ]:
import tomllib

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.sim_utils import generate_time_index
from src.plot_utils import convert_size
from src.stimulus_generator import StimulusGenerator
from src.v1_model import V1Model

In [ ]:
figure_path = '../results/figures/suppl_two'
os.makedirs(figure_path, exist_ok=True)

colormap="coolwarm"
fontsize = 10
figure_size = (89, 89)

figure_size_inches = convert_size(*figure_size)


In [ ]:
class FigureGroundGenerator(StimulusGenerator):

    def __init__(self, parameters):
        super().__init__(parameters)
        self.annulus_diameter = parameters['annulus_diameter']
        self.side_length = parameters['stimulus_side_length']

        stimulus_eccentricity = parameters['stimulus_eccentricity']
        self.xy_offset = np.sqrt(stimulus_eccentricity**2 / 2)
        lower = self.xy_offset - parameters['figure_side_length'] / 2
        upper = lower + parameters['figure_side_length']
        self.figure_bounds = (lower, upper)

    def _check_figure(self, row, col):
        """ 
        Check if the annulus at (row, col) is within the figure bounds.

        Parameters
        ----------
        row : int
            The row index of the annulus center.
        col : int
            The column index of the annulus center.

        Returns
        -------
        bool
            True if the annulus is within the figure bounds, False otherwise.
        """
        x, y = self._pixel_to_position(row, col)

        x_lower = x - self.annulus_diameter / 2
        x_upper = x + self.annulus_diameter / 2
        y_lower = y - self.annulus_diameter / 2
        y_upper = y + self.annulus_diameter / 2

        lower, upper = self.figure_bounds

        if (x_lower >= lower and x_upper <= upper and
            y_lower >= lower and y_upper <= upper):
            return True
        else:
            return False

    def _pixel_to_position(self, row, col):
        """
        Convert pixel indices to position in visual degrees.
        
        Parameters
        ----------
        row : int
            The row index of the pixel.
        col : int
            The column index of the pixel.  
        
        Returns
        -------
        x : float
            The x position in visual degrees.
        y : float
            The y position in visual degrees.
        """
        pixel_size = self.side_length / self.stimulus_resolution
        x = col * pixel_size
        y = row * pixel_size
        return x, y

    def generate(self, scaling_factor, contrast_range, mean_contrast=0.5):
        """
        Generate a stimulus.

        Parameters
        ----------
        scaling_factor : float
            The scaling factor for the grid.
        contrast_range : float
            The range of the contrast.
        mean_contrast : float
            The mean contrast.

        Returns
        -------
        array_like
            The generated stimulus.
        """
        grid = self._get_grid(scaling_factor)
        stimulus = np.ones(
            (self.stimulus_resolution, self.stimulus_resolution)) * 0.5
        indices = np.arange(self.annulus_resolution)
        annulus_half_res = self.annulus_resolution // 2
        for row, col in grid:
            self._check_figure(row, col)
            left, right = row - annulus_half_res, row + annulus_half_res
            down, up = col - annulus_half_res, col + annulus_half_res

            lower_row, upper_row = np.clip([left, right], 0,
                                           self.stimulus_resolution)
            lower_col, upper_col = np.clip([down, up], 0,
                                           self.stimulus_resolution)

            range_row = upper_row - lower_row
            range_col = upper_col - lower_col

            if left < 0:
                row_indices = indices[-range_row:]
            else:
                row_indices = indices[:range_row]

            if down < 0:
                col_indices = indices[-range_col:]
            else:
                col_indices = indices[:range_col]


            # figure region
            if self._check_figure(row, col):
                contrast_factor = np.random.uniform(
                    mean_contrast - contrast_range / 2,
                    mean_contrast + contrast_range / 2)

            # background region
            else:
                contrast_factor = np.random.uniform(
                    mean_contrast - 0.5,
                    mean_contrast + 0.5)

            stimulus[lower_row:upper_row, lower_col:upper_col] = self.annulus[
                row_indices, :][:, col_indices] * contrast_factor + 0.5

        return stimulus

In [ ]:
def load_configurations():
    """
    Load the model, stimulus, simulation, and experiment parameters.

    Returns
    -------
    model_parameters : dict
        The model parameters.
    stimulus_parameters : dict
        The stimulus parameters.
    simulation_parameters : dict
        The simulation parameters.
    """
    parameters = {}
    config_files = ['model', 'stimulus', 'simulation']

    for config_file in config_files:
        with open(f'../config/simulation/{config_file}.toml', 'rb') as f:
            parameters[config_file] = tomllib.load(f)

    return parameters['model'], parameters['stimulus'], parameters['simulation']


def run_experiment(stimulus_parameters, model_parameters, simulation_parameters, gc = 1.1, ch = 0.2575, num_simulations = 5):
    """
    Run the V1 model experiment.

    Parameters
    ----------
    stimulus_parameters : dict
        The stimulus parameters.
    model_parameters : dict
        The model parameters.
    simulation_parameters : dict
        The simulation parameters.
    gc : float
        The grid coarseness.
    ch : float
        The contrast heterogeneity.
    num_simulations : int
        The number of simulations to run.

    Returns
    -------
    figure_sync : np.ndarray
        The synchronization values for the figure region.
    background_sync : np.ndarray
        The synchronization values for the background region.
    """


    # initialize stimulus generator
    stimulus_generator = FigureGroundGenerator(stimulus_parameters)

    # initialize model
    model = V1Model(model_parameters, stimulus_parameters)

    # generate time index
    sync_index, _ = generate_time_index(simulation_parameters)

    # set reference oscillator
    rowscols = int(np.sqrt(model_parameters['num_populations']))
    midpoint = rowscols // 2
    reference = midpoint * rowscols + midpoint

    # initialize phase locking value array
    plv = np.zeros((num_simulations, model_parameters['num_populations']))

    for simulation in range(num_simulations):

        # construct stimulus
        stimulus = stimulus_generator.generate(gc, ch)

        # compute omega
        model.compute_omega(stimulus.flatten())

        # run simulation
        state, _ = model.simulate(simulation_parameters)

        # compute phase differences to reference oscillator
        phase_difference = np.exp(1j * (state - state[:, reference][:, np.newaxis]))

        # compute phase locking values
        plv[simulation, :] = np.abs(np.mean(phase_difference[sync_index, :], axis=0))

    return plv.reshape((-1, rowscols, rowscols))


def save_figure(data, filename):
    sns.set_style('white')
    sns.set_context('paper')
    sns.set_palette('muted')

    plt.Figure(figsize=figure_size_inches)
    im = plt.imshow(data, cmap=colormap, vmin=0, vmax=1, aspect='equal')

    sns.despine()
    plt.axis('off')
    cbar = plt.colorbar(im, orientation='vertical', fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=fontsize)
    cbar.set_label('PLV', fontsize=fontsize)
    plt.tight_layout()
    plt.savefig(filename, format='svg')
    plt.close()


In [ ]:
model_parameters, stimulus_parameters, simulation_parameters = load_configurations()

# add num_time_steps
simulation_parameters['num_time_steps'] = int(
        simulation_parameters['simulation_time'] /
        simulation_parameters['time_step'])

# increase number of populations to accommodate larger stimulus
model_parameters['num_populations'] = 40**2

pix_per_deg = stimulus_parameters['stimulus_resolution'] / stimulus_parameters['stimulus_side_length']

stimulus_parameters['figure_side_length'] = 6.7
stimulus_parameters['stimulus_side_length'] = 9.9
stimulus_parameters['stimulus_resolution'] = int(stimulus_parameters['stimulus_side_length'] * pix_per_deg)
stimulus_parameters['stimulus_num_pixels'] = stimulus_parameters['stimulus_resolution'] ** 2

In [ ]:
plv = run_experiment(stimulus_parameters, model_parameters, simulation_parameters, gc = 1.0, ch = 0.01, num_simulations = 20)
plv_c1 = np.mean(plv, axis=0)

save_figure(plv_c1, os.path.join(figure_path, 'panel_b.svg'))

In [ ]:
plv = run_experiment(stimulus_parameters, model_parameters, simulation_parameters, gc = 1.125, ch = 0.01, num_simulations = 20)
plv_c2 = np.mean(plv, axis=0)

save_figure(plv_c2, os.path.join(figure_path, 'panel_c.svg'))

In [ ]:
plv = run_experiment(stimulus_parameters, model_parameters, simulation_parameters,gc = 1.125, ch = 0.2575, num_simulations = 20)
plv_c3 = np.mean(plv, axis=0)

save_figure(plv_c3, os.path.join(figure_path, 'panel_d.svg'))

In [ ]:
plv = run_experiment(stimulus_parameters, model_parameters, simulation_parameters,gc = 1.25, ch = 0.2575, num_simulations = 20)
plv_c4 = np.mean(plv, axis=0)

save_figure(plv_c4, os.path.join(figure_path, 'panel_e.svg'))